# 持续观察教程和线上结果

一次改动后，平均分变好仍可能掩盖关键问题的退化。本页演示怎样按同一批问题比较结果、阻止不完整的比较进入发布判断，再把离线检查扩展到线上观察。

下面的分数、计数与延迟均为格式示例，用来验证比较逻辑，不代表教程实测或线上流量。教程可先运行自带文件与保存输出检查；线上系统再按需要记录错误、拒答、耗时、费用和访问范围问题。


In [1]:
before = {"questions": 50, "retrieval_failures": 9, "unsupported_answers": 4, "permission_errors": 0}
after = {"questions": 50, "retrieval_failures": 6, "unsupported_answers": 4, "permission_errors": 0}

labels = {"retrieval_failures": "检索失败", "unsupported_answers": "缺少资料依据的回答", "permission_errors": "权限错误"}
for key in labels:
    print(f"{labels[key]}：{before[key]} → {after[key]}")
print("解读：检索失败减少，但无资料支持的回答没变，还不能说整体问题已解决。")

检索失败：9 → 6
缺少资料依据的回答：4 → 4
权限错误：0 → 0
解读：检索失败减少，但无资料支持的回答没变，还不能说整体问题已解决。


示例中检索失败从 9 次减到 6 次，但无依据回答仍为 4 次。它提示我们分别看检索与回答；下面进一步按稳定问题 ID 对齐结果，检查是否遗漏失败样本。


## 原理：先对齐问题，再判断改动

离线比较固定问题集、资料版本、返回数量和评分规则。先拒绝空或非法 `case_id`、非有限分数，再检查重复 ID 和完整集合中的 `missing`、`added`。集合不同或存在重复时，不计算交集上的“平均提升”；只有同一且完整的案例集才计算逐题改善、退化与持平。

下面的 `compare_runs` 返回对齐信息，`release_check` 根据最低均分和最多退化题数判断。阈值需根据自己已核对的问题设定；这里仅用小样本验证逻辑，同时演示线上延迟、错误率和拒答率的汇总格式。


In [2]:
from collections import Counter
from collections.abc import Mapping
from math import isfinite
from numbers import Real
from statistics import mean, median

def _index_cases(rows, id_key, score_key, side):
    rows = list(rows)
    validated = []
    for index, row in enumerate(rows):
        if not isinstance(row, Mapping):
            raise TypeError(f"{side}[{index}] 必须是 mapping")
        if id_key not in row:
            raise ValueError(f"{side}[{index}] 缺少 {id_key!r}")
        case_id = row[id_key]
        if not isinstance(case_id, str) or not case_id.strip() or case_id != case_id.strip():
            raise ValueError(f"{side}[{index}] 的 {id_key} 必须是无首尾空白的非空字符串")
        if score_key not in row:
            raise ValueError(f"{side}[{index}] 缺少 {score_key!r}")
        score = row[score_key]
        if isinstance(score, bool) or not isinstance(score, Real) or not isfinite(float(score)):
            raise ValueError(f"{side}[{index}] 的 {score_key} 必须是有限实数")
        validated.append((case_id, row))
    counts = Counter(case_id for case_id, _ in validated)
    duplicate_ids = sorted((case_id for case_id, count in counts.items() if count > 1), key=str)
    unique_ids = set(counts)
    indexed = {case_id: row for case_id, row in validated if counts[case_id] == 1}
    return indexed, unique_ids, duplicate_ids

def compare_runs(before_rows, after_rows, key="score", id_key="case_id"):
    """按稳定的 case_id 对齐；案例集合不完整时不计算逐例回归。"""
    before, before_ids, duplicate_before = _index_cases(before_rows, id_key, key, "baseline")
    after, after_ids, duplicate_after = _index_cases(after_rows, id_key, key, "after")
    missing = sorted(before_ids - after_ids, key=str)
    added = sorted(after_ids - before_ids, key=str)
    common = sorted(before_ids & after_ids, key=str)
    case_set_match = not missing and not added
    comparison_complete = case_set_match and not duplicate_before and not duplicate_after
    compared_ids = common if comparison_complete else []
    deltas = [after[case_id][key] - before[case_id][key] for case_id in compared_ids]
    return {
        "n": len(compared_ids),
        "n_before": len(before_ids),
        "n_after": len(after_ids),
        "overlap_n": len(common),
        "compared_ids": compared_ids,
        "before_ids": sorted(before_ids, key=str),
        "baseline_ids": sorted(before_ids, key=str),
        "after_ids": sorted(after_ids, key=str),
        "missing": missing,
        "added": added,
        "duplicate_before": duplicate_before,
        "duplicate_baseline": duplicate_before,
        "duplicate_after": duplicate_after,
        "case_set_match": case_set_match,
        "comparison_complete": comparison_complete,
        "mean_before": mean(before[case_id][key] for case_id in compared_ids) if compared_ids else None,
        "mean_after": mean(after[case_id][key] for case_id in compared_ids) if compared_ids else None,
        "wins": sum(delta > 0 for delta in deltas),
        "regressions": sum(delta < 0 for delta in deltas),
        "ties": sum(delta == 0 for delta in deltas),
    }

def summarize_online_events(events):
    latencies = sorted(float(e["latency_ms"]) for e in events if e.get("latency_ms") is not None)
    if not latencies: return {"n": 0}
    p95_index = min(len(latencies) - 1, int(len(latencies) * 0.95))
    return {"n": len(latencies), "p50_ms": median(latencies), "p95_ms": latencies[p95_index],
            "error_rate": sum(bool(e.get("error")) for e in events) / len(events),
            "refusal_rate": sum(bool(e.get("refused")) for e in events) / len(events)}

def release_check(summary, minimum_mean, maximum_regressions):
    """CI 门禁：完整且无重复的同一案例集才允许按阈值放行。"""
    return (
        summary.get("case_set_match", False)
        and summary.get("comparison_complete", False)
        and summary.get("n", 0) > 0
        and not summary.get("duplicate_baseline", summary.get("duplicate_before"))
        and not summary.get("duplicate_after")
        and not summary.get("missing")
        and not summary.get("added")
        and summary.get("mean_after") is not None
        and summary["mean_after"] >= minimum_mean
        and summary["regressions"] <= maximum_regressions
    )

sample_before = [{"case_id": "q1", "score": 0.70}, {"case_id": "q2", "score": 0.90}]
sample_after = [{"case_id": "q2", "score": 0.90}, {"case_id": "q1", "score": 0.80}]
sample_summary = compare_runs(sample_before, sample_after)
assert sample_summary["missing"] == [] and sample_summary["added"] == []
assert sample_summary["duplicate_before"] == [] and sample_summary["duplicate_after"] == []
sample_after_missing = [{"case_id": "q1", "score": 0.80}]
sample_missing_summary = compare_runs(sample_before, sample_after_missing)
assert sample_missing_summary["missing"] == ["q2"]
assert not release_check(sample_missing_summary, 0.0, 0)
sample_added_summary = compare_runs(sample_before, sample_after + [{"case_id": "q3", "score": 0.80}])
sample_duplicate_summary = compare_runs(sample_before, sample_after + [{"case_id": "q1", "score": 0.80}])
assert sample_added_summary["added"] == ["q3"] and not release_check(sample_added_summary, 0.0, 0)
assert sample_duplicate_summary["duplicate_after"] == ["q1"] and not release_check(sample_duplicate_summary, 0.0, 0)

def assert_invalid_rows(before_rows, after_rows):
    try:
        compare_runs(before_rows, after_rows)
    except (TypeError, ValueError):
        return
    raise AssertionError("非法 case_id 或 score 不得进入 CI 比较")

for invalid_id in (None, "", "   "):
    assert_invalid_rows([{"case_id": invalid_id, "score": 0.70}], sample_after[:1])
for invalid_score in (None, True, "0.70", float("nan"), float("inf"), float("-inf")):
    assert_invalid_rows([{"case_id": "q1", "score": invalid_score}], sample_after[:1])
sample_online = summarize_online_events([
    {"latency_ms": 100, "error": False, "refused": False},
    {"latency_ms": 200, "error": True, "refused": True},
])
print(
    f"示例对照：完整比较 {sample_summary['n']} 题，平均分 {sample_summary['mean_before']:.2f} → {sample_summary['mean_after']:.2f}，"
    f"提升 {sample_summary['wins']} 题，退化 {sample_summary['regressions']} 题，持平 {sample_summary['ties']} 题。"
)
print(
    f"案例检查：missing={sample_summary['missing']}，added={sample_summary['added']}，"
    f"重复 baseline={sample_summary['duplicate_before']}，after={sample_summary['duplicate_after']}，"
    f"集合一致={sample_summary['case_set_match']}；after 少一题示例 missing={sample_missing_summary['missing']}，"
    f"门禁{'通过' if release_check(sample_summary, 0.75, 0) else '未通过'} / "
    f"少题门禁{'通过' if release_check(sample_missing_summary, 0.0, 0) else '未通过'}。"
)
print("输入校验：空/非法 case_id 与 None、布尔、字符串、NaN、±inf 分数均已拒绝。")
print(
    f"线上示例：共 {sample_online['n']} 次请求，中位耗时 {sample_online['p50_ms']:.0f} 毫秒，"
    f"较慢请求耗时 {sample_online['p95_ms']:.0f} 毫秒，错误率 {sample_online['error_rate']:.2f}，"
    f"拒答率 {sample_online['refusal_rate']:.2f}；"
    f"合并前检查{'通过' if release_check(sample_summary, 0.75, 0) else '未通过'}。"
)

示例对照：完整比较 2 题，平均分 0.80 → 0.85，提升 1 题，退化 0 题，持平 1 题。
案例检查：missing=[]，added=[]，重复 baseline=[]，after=[]，集合一致=True；after 少一题示例 missing=['q2']，门禁通过 / 少题门禁未通过。
输入校验：空/非法 case_id 与 None、布尔、字符串、NaN、±inf 分数均已拒绝。
线上示例：共 2 次请求，中位耗时 150 毫秒，较慢请求耗时 200 毫秒，错误率 0.50，拒答率 0.50；合并前检查通过。


## 运行本地检查

按[教程首页](../README.md#运行准备)进入 C7 根目录并安装 `requirements-c7.txt` 后，在同一目录运行：

```bash
python scripts/check_tutorial.py
```

脚本检查 Notebook、链接、案例和保存结果，不调用评估模型。本页示例比较器只验证对齐和判断逻辑，实际质量仍需结合失败问题、人工反馈和逐题依据核对。


## 扩展一：比较 Prompt 版本

两版提示词沿用前面的案例对齐检查，并固定资料、返回数量、模型和评分规则。下面展示 Prompt 文本、运行配置与结果行的结构，不调用模型。

普通教学保存实际输入和输出。长期回归或跨版本比较再记录数据、索引、模型、Prompt 与评审配置；线上记录使用运行编号和必要证据引用，按需求增加延迟、检索轮数、token、费用、错误/拒答与用户反馈。保留脱敏摘要，避免无关原文进入日志。


In [3]:
PROMPT_V1 = "根据上下文回答问题；没有依据时请明确说无法回答。"
PROMPT_V2 = "严格根据上下文回答，逐条核对主要结论；没有依据时请明确说无法回答，并给出可核对的引用。"
PROMPT_VERSION_1 = "prompt-v1"
PROMPT_VERSION_2 = "prompt-v2"

def build_run_manifest(run_id, *, dataset_version, index_version, model_version, prompt_version,
                       top_k, evaluator_version=None):
    """保存一次可复查运行所需的版本字段，不保存问题或资料原文。"""
    return {
        "run_id": run_id,
        "dataset_version": dataset_version,
        "index_version": index_version,
        "model_version": model_version,
        "prompt_version": prompt_version,
        "top_k": top_k,
        "evaluator_version": evaluator_version,
    }

def compare_prompt_versions(before_rows, after_rows):
    """按 case_id 对齐 Prompt 前后分数；分数应来自同一评分协议。"""
    summary = compare_runs(before_rows, after_rows, key="score")
    summary["prompt_before"] = PROMPT_VERSION_1
    summary["prompt_after"] = PROMPT_VERSION_2
    return summary

example_manifest = build_run_manifest("example-only", dataset_version="教程问题集示例",
    index_version="随附 BGE 向量库示例", model_version="未调用模型的格式示例", prompt_version=PROMPT_VERSION_1, top_k=4)
assert example_manifest["prompt_version"] == PROMPT_VERSION_1
prompt_summary = compare_prompt_versions(
    [{"case_id": "q1", "score": 0.70}],
    [{"case_id": "q1", "score": 0.80}],
)
print(
    f"示例记录：同一份问题集和向量库下，完整比较 {prompt_summary['n']} 题，调整提示词后平均分从 {prompt_summary['mean_before']:.2f} 变为 {prompt_summary['mean_after']:.2f}，"
    f"提升 {prompt_summary['wins']} 题，退化 {prompt_summary['regressions']} 题，持平 {prompt_summary['ties']} 题；集合一致={prompt_summary['case_set_match']}。"
)
# 真实运行时还要把模型、索引、评估集和评分协议固定并保存。


示例记录：同一份问题集和向量库下，完整比较 1 题，调整提示词后平均分从 0.70 变为 0.80，提升 1 题，退化 0 题，持平 0 题；集合一致=True。


## 扩展二：五类工具接入

RAG 三元组（Context Relevance、Groundedness、Answer Relevance）适合作为跨工具的共同语言，但三项都不必由同一个框架完成。先把问题、检索证据、回答和判定结果放进稳定的数据契约，再按运行位置选择工具。

| 类别（代表路径） | 主要目标 | 输入（input） | 输出（output） | 证据与元数据 | 适用场景 | 边界 |
| --- | --- | --- | --- | --- | --- | --- |
| 本地确定性指标（项目脚本/纯 Python） | 快速检查检索和协议是否正确 | `query`、带 `rank/evidence_id` 的证据列表、可选 qrels、`answer` 和引用 ID | `Hit@k`、`Recall@k`、`MRR`、引用 ID 有效率、错误/拒答计数等固定数值 | 保留 evidence ID、排名、来源引用；配置只保留会影响比较的字段 | Notebook 教学、smoke check、快速回归、无密钥 CI | 词面重合或排名指标不能证明语义忠实、流畅度或用户满意度 |
| LLM Judge（自建 rubric；指标库适配层） | 检查切题、依据充分、声明覆盖等语义属性 | `case_id`、`query`、带 ID 的 evidence、`answer`；离线时可选 `reference_claims` | 严格结构化的 `score`/`label`、简短 `reason`、支持/不支持的 claim 及 evidence ID | 记录 rubric/prompt、采样策略和校准集；只引用输入中存在的 evidence ID | 语义相关性、Groundedness、复杂回答的覆盖检查 | 有成本、随机性、位置/长度偏差；解析失败应报错，不能用默认分数替代人工校准 |
| 离线评估框架（[Ragas](https://docs.ragas.io/en/stable/getstarted/) 等） | 复用 Context Precision、Faithfulness、Answer/Response Relevancy 等评估实现 | 映射后的 `user_input/query`、`retrieved_contexts/evidence`、`response/answer`，按指标提供 reference | 每项指标分数和逐样本结果 | 仍要保留 canonical case/evidence ID、原始输入、指标配置和实际评审模型 | 批量离线评估、与项目自定义指标交叉检查 | 字段和 API 会随版本变化；框架分数不替代 qrels、原文核对、人工校准或安全审核 |
| CI 测试（pytest 风格；DeepEval 仅接入示意） | 在合并前阻止可复现的质量回退 | 固定评估集、候选与 baseline 的逐题指标、门禁策略 | `pass/fail`、逐题 delta、回退 case 列表、门禁原因 | 保存数据集/证据范围、指标协议和阈值；跨版本时再保存相关组件 ID | PR 检查、发布门禁、变更后的回归护栏 | 不是线上用户分布；LLM Judge 门禁可能昂贵或不稳定，先用确定性检查缩小范围 |
| tracing / dashboard（TruLens 类反馈/仪表盘，或 OpenTelemetry + 可观测平台） | 查看一次请求的链路、趋势和失败归因 | `trace_id`、各阶段 span 的 input/output 引用、evidence ID、状态和资源用量 | span 时间线、p50/p95 延迟、错误/拒答/成本趋势及抽样样本 | 至少有 `run_id/trace_id`、stage、timestamp、latency、token/cost（若可得）、脱敏版本；默认不存整段敏感原文 | 线上监控、漂移发现、定位检索或生成阶段瓶颈 | 需要存储、采样和隐私治理；dashboard 展示相关性，不等于事实正确性或安全审核 |

本 Notebook 当前环境未安装 Ragas、TruLens 或 DeepEval。它们在此只表示接入示意和选择路径：不导入、不调用，也不保存伪运行输出。接入时让项目适配层负责字段映射，具体第三方 API 以实际锁定的依赖和文档为准，不把框架 API 当成数据契约。

### 五类接入共用的数据记录

统一的评估样本至少有 `{case_id, query, evidence, answer}`。`evidence` 的每项至少包含 `{evidence_id, quote, rank, source_ref}`；`source_ref` 可以是页码、文档 ID 或脱敏 URI。`reference_claims`、相关 evidence ID 和人工标签只供离线评估使用，不能在生成回答之前偷偷作为检索输入。

适配层将同一份记录映射给上表中的工具。模型评审要检查分数范围、必填字段和 evidence ID，再用人工已判定样本校准；CI 使用完整同集的逐题结果并在失败时返回非零状态；tracing 按阶段传递必要引用、状态和资源事件。框架 API 和字段名以锁定版本为准，项目自己的问题与证据身份保持稳定。

普通教学运行只需保存本单元实际展示的输入和输出，不强制记录模型版本、执行时间或源码 revision。只有长期回归、线上监控或跨版本比较，才记录解释差异所需的最少字段：`run_id`、`case_id`、相关数据/索引/Prompt/模型标识、指标或 rubric 版本；线上再按需增加 `trace_id`、时间、延迟、token、费用、错误/拒答和脱敏策略。工具选择的顺序可以是“先跑确定性护栏 → 按采样策略调用 Judge → 用 CI 做门禁 → 用 tracing/dashboard 观察线上趋势”，而不是把一次分数当成完整质量结论。


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[执行端到端验收](端到端验收.ipynb)

